In [2]:
import os
import csv
import pandas as pd
import re
import subprocess
import numpy as np
import scanpy as sc
import scipy.stats as stats
from scipy.stats import combine_pvalues

new_dir = "/home/jingqi/RNALocateV3.0"
os.chdir(new_dir)
os.getcwd()

'/home/jingqi/RNALocateV3.0'

### crceating usable fasta file (can be neglected)

In [25]:
input_fasta = "Data/Raw/3UTR_clean.txt"
safe_fasta = "Data/Raw/3UTR_safe.txt"

print(f"Reading from: {input_fasta}")
print(f"Writing safe output to: {safe_fasta}")

seq_count = 0
with open(input_fasta, "r") as f_in, open(safe_fasta, "w") as f_out:
    for line in f_in:
        line = line.strip()
        if not line:
            continue
            
        if line.startswith(">"):
            header = line.split()[0]
            if len(header) > 50:
                header = header[:50]
            f_out.write(header + "\n")
            seq_count += 1
        else:
            clean_seq = re.sub(r'[^A-Za-z]', '', line)
            if clean_seq:
                f_out.write(clean_seq + "\n")

print(f"Successfully generated sanitized FASTA file containing {seq_count} sequences.")

Reading from: Data/Raw/3UTR_clean.txt
Writing safe output to: Data/Raw/3UTR_safe.txt
Successfully generated sanitized FASTA file containing 2432 sequences.


## Processing MEME findings

In [49]:
meme_output_dir = "Data/MEME_New/meme_results"
master_meme_file = "Data/MEME_New/filtered_motifs_fimo.meme"

label_names = [
    "extracellular", "membrane", "mitochondrion", "nucleolus", "nucleoplasm", "nucleus", "ribosome",
    "chromatin", "cytoplasm", "cytosol", "ER"
]

id_map = {
    "ER": "ER",
    "extracellular": "EX",
    "membrane": "mem", 
    "ribosome": "ribo",
    "mitochondrion": "mito",
    "chromatin": "chrom",
    "nucleolus": "nucl",
    "nucleoplasm": "nucp",
    "nucleus": "nuc",
    "cytoplasm": "cyp",
    "cytosol": "cys"
}

E_VALUE_THRESH = 0.05
MIN_SITES = 25 

master_motifs = []
meme_header = ""

for class_name in label_names:
    bucket = 'positive'
    meme_file = f"{meme_output_dir}/{class_name}_{bucket}/meme.txt"
    
    if not os.path.exists(meme_file):
        continue
        
    with open(meme_file, 'r') as f:
        lines = f.readlines()
        
    if not meme_header:
        header_lines = []
        for line in lines:
            if line.startswith("MOTIF "):
                break
            header_lines.append(line)
        meme_header = "".join(header_lines)
        
    i = 0
    while i < len(lines):
        if lines[i].startswith("MOTIF "):
            parts = lines[i].strip().split()
            orig_motif_id = parts[1]
            
            j = i + 1
            matrix_start = -1
            w, sites, e_val = 0, 0, 1.0
            
            while j < len(lines) and not lines[j].startswith("MOTIF "):
                if "letter-probability matrix:" in lines[j]:
                    matrix_start = j
                    m_w = re.search(r'w=\s*(\d+)', lines[j])
                    m_s = re.search(r'nsites=\s*(\d+)', lines[j])
                    m_e = re.search(r'E=\s*([0-9\.eE+-]+)', lines[j])
                    
                    if m_w: w = int(m_w.group(1))
                    if m_s: sites = int(m_s.group(1))
                    if m_e: e_val = float(m_e.group(1))
                    break
                j += 1
                
            if matrix_start != -1 and e_val <= E_VALUE_THRESH and sites >= MIN_SITES:
                clean_prefix = id_map.get(class_name, class_name)
                new_motif_id = f"{clean_prefix}_{orig_motif_id}"
                
                motif_str = f"MOTIF {new_motif_id}\n"
                motif_str += lines[matrix_start]
                
                for k in range(1, w + 1):
                    if matrix_start + k < len(lines):
                        motif_str += lines[matrix_start + k]
                motif_str += "\n"
                master_motifs.append(motif_str)
            
            i = j
        else:
            i += 1

with open(master_meme_file, 'w') as f:
    f.write(meme_header)
    for motif in master_motifs:
        f.write(motif)

print(f"Extraction complete. {len(master_motifs)} valid positive motif matrices saved to {master_meme_file}.")

Extraction complete. 47 valid positive motif matrices saved to Data/MEME_New/filtered_motifs_fimo.meme.


## Mapping

In [50]:
input_file = "Data/Raw/3UTR_clean.txt"
output_file = "Data/Raw/gene_3UTR_matrix.csv"

print(f"Reading from: {input_file}")

gene_to_transcripts = {}

with open(input_file, 'r') as f:
    for line in f:
        if line.startswith(">"):
            clean_line = line.strip()[1:] 
            parts = clean_line.split('|')
            
            if len(parts) >= 2:
                # Corrected index mapping
                gene_symbol = parts[0]
                transcript_id = parts[1]
                
                if gene_symbol not in gene_to_transcripts:
                    gene_to_transcripts[gene_symbol] = []
                    
                # Prevent identical transcript duplicates
                if transcript_id not in gene_to_transcripts[gene_symbol]:
                    gene_to_transcripts[gene_symbol].append(transcript_id)

max_transcripts = 0
rows = []

for gene, transcripts in gene_to_transcripts.items():
    if len(transcripts) > max_transcripts:
        max_transcripts = len(transcripts)
    
    row = [gene] + transcripts
    rows.append(row)

print(f"Found {len(rows)} unique genes.")
print(f"Max transcripts for one gene: {max_transcripts}")

with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)
    
    header = ["Gene_Symbol"] + [f"Transcript_{i+1}" for i in range(max_transcripts)]
    writer.writerow(header)
    
    writer.writerows(rows)


Reading from: Data/Raw/3UTR_clean.txt
Found 713 unique genes.
Max transcripts for one gene: 18


## Run FIMO

In [51]:
input_fasta = "Data/Raw/3UTR_safe.txt"
bfile_path = "Data/MEME_New/background_model.bfile"
master_meme_file = "Data/MEME_New/filtered_motifs_fimo.meme"
fimo_output_dir = "Data/FIMO_New/Aginst_MEME"
fimo_path = "/home/jingqi/miniconda3/envs/rnalocate_old/bin/fimo"

os.makedirs(fimo_output_dir, exist_ok=True)

fimo_cmd = [
    fimo_path,
    "--oc", fimo_output_dir,
    "--thresh", "0.0001", 
    "--norc",
    "--bgfile", bfile_path,
    master_meme_file,
    input_fasta
]

print("Executing FIMO with de novo motifs...")
subprocess.run(fimo_cmd, check=True)
print("FIMO complete.")

Executing FIMO with de novo motifs...


Using motif +EX_1 of width 8.
Computing q-values.
Estimating pi_0 from a uniformly sampled set of 10000 p-values.
Estimating pi_0.
Estimated pi_0=0.982766
Using motif +mem_1 of width 10.
Computing q-values.
Estimating pi_0 from a uniformly sampled set of 10000 p-values.
Estimating pi_0.
Estimated pi_0=0.940843
Using motif +mem_2 of width 10.
Computing q-values.
Estimating pi_0 from a uniformly sampled set of 10000 p-values.
Estimating pi_0.
Estimated pi_0=0.941227
Using motif +mem_3 of width 10.
Computing q-values.
Estimating pi_0 from a uniformly sampled set of 10000 p-values.
Estimating pi_0.
Estimated pi_0=0.943721
Using motif +mem_4 of width 10.
Computing q-values.
Estimating pi_0 from a uniformly sampled set of 10000 p-values.
Estimating pi_0.
Estimated pi_0=0.987755
Using motif +mem_5 of width 10.
Computing q-values.
Estimating pi_0 from a uniformly sampled set of 10000 p-values.
Estimating pi_0.
Estimated pi_0=0.9704
Using motif +mem_7 of width 10.
Computing q-values.
Estimating

FIMO complete.


## Filter by q-value

In [52]:
fimo_txt_file = "Data/FIMO_New/Aginst_MEME/fimo.txt"
output_filtered_csv = "Data/FIMO_New/Aginst_MEME_file/filtered_fimo_results.csv"

fimo_cols = [
    "motif_id", "sequence_name", "start", "stop", "strand", 
    "score", "p-value", "q-value", "matched_sequence"
]

fimo_df = pd.read_csv(
    fimo_txt_file, 
    sep='\t', 
    comment='#', 
    header=None, 
    names=fimo_cols
)

fimo_df['q-value'] = pd.to_numeric(fimo_df['q-value'], errors='coerce')

filtered_fimo = fimo_df[fimo_df['q-value'] < 0.05].copy()

filtered_fimo.to_csv(output_filtered_csv, index=False)

print(f"Raw scanning events: {len(fimo_df)}")
print(f"High-confidence events (q < 0.05): {len(filtered_fimo)}")
print(f"Filtered results saved to {output_filtered_csv}")

Raw scanning events: 184396
High-confidence events (q < 0.05): 133914
Filtered results saved to Data/FIMO_New/Aginst_MEME_file/filtered_fimo_results.csv


## Merge the same hits

In [53]:
fimo_filtered_csv = "Data/FIMO_New/Aginst_MEME_file/filtered_fimo_results.csv"
transcript_level_csv = "Data/FIMO_New/Aginst_MEME_file/transcript_level.csv"

df = pd.read_csv(fimo_filtered_csv)

def fisher_combine(p_array):
    p_vals = np.asarray(p_array)
    p_vals = np.where(p_vals == 0, 1e-300, p_vals) 
    if len(p_vals) == 1:
        return float(p_vals[0])
    return combine_pvalues(p_vals, method='fisher')[1]

agg_df = df.groupby(['sequence_name', 'motif_id']).agg(
    site_count=('p-value', 'count'),
    transcript_p_value=('p-value', fisher_combine)
).reset_index()

transcript_p_thresh = 1e-6
final_df = agg_df[agg_df['transcript_p_value'] < transcript_p_thresh].copy()

split_names = final_df['sequence_name'].str.split('|', expand=True)
final_df['Gene'] = split_names[0]
final_df['Transcript'] = split_names[1]

cols = ['Gene', 'Transcript', 'sequence_name', 'motif_id', 'site_count', 'transcript_p_value']
final_df = final_df[cols]

final_df.to_csv(transcript_level_csv, index=False)

print(f"Unique Transcript-Motif pairs analyzed: {len(agg_df)}")
print(f"Highly significant transcript-level events (p < {transcript_p_thresh}): {len(final_df)}")
print(f"Data saved to {transcript_level_csv}")

Unique Transcript-Motif pairs analyzed: 31304
Highly significant transcript-level events (p < 1e-06): 19529
Data saved to Data/FIMO_New/Aginst_MEME_file/transcript_level.csv


## Hits variance

In [54]:
variance_matrix_csv = "Data/FIMO_New/Aginst_MEME_file/Isoform_Variance_Matrix.csv"

matrix = final_df.pivot_table(
    index=['Gene', 'Transcript'], 
    columns='motif_id', 
    values='site_count', 
    fill_value=0
).reset_index()

matrix.columns.name = None
motif_columns = [col for col in matrix.columns if col not in ['Gene', 'Transcript']]

matrix[motif_columns] = (matrix[motif_columns] > 0).astype(int)

def has_binary_variance(group):
    if len(group) == 1:
        return False
    unique_profiles = group[motif_columns].drop_duplicates()
    return len(unique_profiles) > 1

dynamic_genes_matrix = matrix.groupby('Gene').filter(has_binary_variance)
dynamic_genes_matrix.to_csv(variance_matrix_csv, index=False)

print(f"Initial transcripts analyzed: {len(matrix)}")
print(f"Transcripts retained after dropping uniform genes: {len(dynamic_genes_matrix)}")
print(f"Total unique genes demonstrating binary motif variance: {dynamic_genes_matrix['Gene'].nunique()}")
print(f"Matrix saved to {variance_matrix_csv}")

Initial transcripts analyzed: 1647
Transcripts retained after dropping uniform genes: 934
Total unique genes demonstrating binary motif variance: 259
Matrix saved to Data/FIMO_New/Aginst_MEME_file/Isoform_Variance_Matrix.csv


## Discard low expression, from aData

In [4]:
adata = sc.read_h5ad("/home/jingqi/isoforms/adata_thresholded.h5ad")

In [6]:
import scipy.sparse as sp

cell_type_col = "assignments" 
CPM_THRESHOLD = 15

variance_matrix_csv = "Data/FIMO_New/Aginst_MEME_file/Isoform_Variance_Matrix.csv"
filtered_variance_csv = "Data/FIMO_New/Aginst_MEME_file/ExpressionFiltered_IsoVar_Matrix.csv"

matrix_df = pd.read_csv(variance_matrix_csv)

print("Reversing log1p transformation...")
if sp.issparse(adata.X):
    raw_counts = adata.X.expm1()
else:
    raw_counts = np.expm1(adata.X)

print(f"Pseudo-bulking across {cell_type_col}...")
unique_cell_types = adata.obs[cell_type_col].unique()

cell_type_sums = []
for ct in unique_cell_types:
    cells_mask = (adata.obs[cell_type_col] == ct).values
    ct_sum = raw_counts[cells_mask].sum(axis=0)
    cell_type_sums.append(np.asarray(ct_sum).flatten())

pseudo_bulk_matrix = np.vstack(cell_type_sums)

print("Calculating CPM...")
library_sizes = pseudo_bulk_matrix.sum(axis=1, keepdims=True)
library_sizes[library_sizes == 0] = 1 
cpm_matrix = (pseudo_bulk_matrix / library_sizes) * 1e6

cpm_df = pd.DataFrame(
    cpm_matrix, 
    index=unique_cell_types, 
    columns=adata.var_names
)

print(f"Filtering transcripts with < {CPM_THRESHOLD} CPM...")
max_cpm_per_transcript = cpm_df.max(axis=0)
expressed_transcripts = max_cpm_per_transcript[max_cpm_per_transcript >= CPM_THRESHOLD].index
expressed_transcripts_clean = expressed_transcripts.str.split('.').str[0]

matrix_df['Clean_Transcript'] = matrix_df['Transcript'].str.split('.').str[0]

initial_count = len(matrix_df)
expressed_matrix_df = matrix_df[matrix_df['Clean_Transcript'].isin(expressed_transcripts_clean)].copy()

expressed_matrix_df.to_csv(filtered_variance_csv, index=False)

print(f"Initial transcripts in variance matrix: {initial_count}")
print(f"Transcripts passing > {CPM_THRESHOLD} CPM in at least one cell type: {len(expressed_matrix_df)}")
print(f"Filtered matrix saved to {filtered_variance_csv}")

Reversing log1p transformation...
Pseudo-bulking across assignments...
Calculating CPM...
Filtering transcripts with < 15 CPM...
Initial transcripts in variance matrix: 934
Transcripts passing > 15 CPM in at least one cell type: 592
Filtered matrix saved to Data/FIMO_New/Aginst_MEME_file/ExpressionFiltered_IsoVar_Matrix.csv


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

COSINE_THRESHOLD = 0.80

# Strip the decimal version numbers from the cpm_df columns to match the clean transcripts
cpm_df.columns = cpm_df.columns.astype(str).str.split('.').str[0]

gene_min_sim_records = []
transcript_avg_sim_records = []
divergent_genes = []

for gene, group in expressed_matrix_df.groupby('Gene'):
    transcripts = group['Clean_Transcript'].values
    
    if len(transcripts) < 2:
        continue
        
    gene_cpm_matrix = cpm_df[transcripts].values 
    total_gene_cpm = gene_cpm_matrix.sum(axis=1)
    
    valid_mask = total_gene_cpm > 0 
    
    if valid_mask.sum() == 0:
        continue
        
    valid_cpm = gene_cpm_matrix[valid_mask]
    valid_totals = total_gene_cpm[valid_mask]
    
    fraction_matrix = valid_cpm / valid_totals[:, None]
    transcript_vectors = fraction_matrix.T 
    
    sim_matrix = cosine_similarity(transcript_vectors)
    upper_tri = sim_matrix[np.triu_indices(len(transcripts), k=1)]
    
    min_gene_sim = np.min(upper_tri)
    
    if min_gene_sim < COSINE_THRESHOLD:
        divergent_genes.append(gene)
        gene_min_sim_records.append({'Gene': gene, 'Min_Gene_Similarity': min_gene_sim})
        
        for i, transcript in enumerate(transcripts):
            all_sims_for_transcript = sim_matrix[i]
            other_isoform_sims = np.delete(all_sims_for_transcript, i)
            avg_isoform_sim = np.mean(other_isoform_sims)
            
            transcript_avg_sim_records.append({
                'Clean_Transcript': transcript, 
                'Avg_Isoform_Similarity': avg_isoform_sim
            })

print(f"Total multi-transcript genes evaluated: {expressed_matrix_df['Gene'].nunique()}")
print(f"Genes with transcript cosine similarity < {COSINE_THRESHOLD}: {len(divergent_genes)}")

gene_metrics_df = pd.DataFrame(gene_min_sim_records)
transcript_metrics_df = pd.DataFrame(transcript_avg_sim_records)

final_expression_divergent_df = expressed_matrix_df[expressed_matrix_df['Gene'].isin(divergent_genes)].copy()

if not gene_metrics_df.empty:
    final_expression_divergent_df = pd.merge(final_expression_divergent_df, gene_metrics_df, on='Gene', how='left')
    final_expression_divergent_df = pd.merge(final_expression_divergent_df, transcript_metrics_df, on='Clean_Transcript', how='left')

final_expression_divergent_df.to_csv(filtered_variance_csv, index=False)
print(f"Metrics attached and matrix overwritten at: {filtered_variance_csv}")

Total multi-transcript genes evaluated: 251
Genes with transcript cosine similarity < 0.8: 127
Metrics attached and matrix overwritten at: Data/FIMO_New/Aginst_MEME_file/ExpressionFiltered_IsoVar_Matrix.csv


## Annotate with SCL

In [10]:
variance_matrix_csv = "Data/FIMO_New/Aginst_MEME_file/ExpressionFiltered_IsoVar_Matrix.csv"
probabilities_file = "Data/Main/Probabilities.csv"
output_file = "Data/FIMO_New/Aginst_MEME_file/Functional_Zipcode_Candidates.csv"

matrix_df = pd.read_csv(filtered_variance_csv)
prob_df = pd.read_csv(probabilities_file)

prob_df['Clean_Transcript'] = prob_df['sequence_id'].str.split('.').str[0]

motif_columns = [c for c in matrix_df.columns if '_' in c] 

def get_motif_profile(row):
    bound_motifs = [m for m in motif_columns if row[m] == 1]
    return ", ".join(bound_motifs) if bound_motifs else "None"

matrix_df['Motif_Profile'] = matrix_df.apply(get_motif_profile, axis=1)

merged_df = pd.merge(matrix_df, prob_df, on='Clean_Transcript', how='inner')

def get_adjusted_scl_profile(row):
    locations = []
    
    nucleus_cols = ['chromatin_prob', 'nucleolus_prob', 'nucleoplasm_prob', 'nucleus_prob']
    if any(row.get(col, 0) > 0.75 for col in nucleus_cols):
        locations.append('Nucleus')
        
    cyto_cols = ['cytoplasm_prob', 'cytosol_prob']
    if any(row.get(col, 0) > 0.75 for col in cyto_cols):
        locations.append('Cytoplasm')
        
    independent_cols = {
        'endoplasmic reticulum_prob': 'Endoplasmic reticulum',
        'extracellular region_prob': 'Extracellular region',
        'membrane_prob': 'Membrane',
        'mitochondrion_prob': 'Mitochondrion',
        'ribosome_prob': 'Ribosome'
    }
    
    for col, name in independent_cols.items():
        if row.get(col, 0) > 0.75:
            locations.append(name)
            
    return ", ".join(locations) if locations else "Unclassified"

merged_df['Adjusted_SCL_Profile'] = merged_df.apply(get_adjusted_scl_profile, axis=1)

def has_adjusted_scl_variance(group):
    unique_scls = group['Adjusted_SCL_Profile'].drop_duplicates()
    return len(unique_scls) > 1

final_candidates = merged_df.groupby('Gene').filter(has_adjusted_scl_variance)

output_cols = [
    'Gene', 
    'Transcript', 
    'Avg_Isoform_Similarity', 
    'Motif_Profile', 
    'Adjusted_SCL_Profile', 
    'Min_Gene_Similarity'
]

final_output = final_candidates[output_cols].copy()

final_output.sort_values(by=['Min_Gene_Similarity', 'Gene', 'Avg_Isoform_Similarity'], ascending=[True, True, True], inplace=True)

final_output.to_csv(output_file, index=False)

print(f"Total transcripts inputted from variance matrix: {len(matrix_df)}")
print(f"Transcripts successfully mapped to prediction file: {len(merged_df)}")
print(f"Transcripts retained after dropping uniform macro-SCL genes: {len(final_output)}")
print(f"Total high-confidence candidate genes: {final_output['Gene'].nunique()}")
print(f"Final dataset saved to: {output_file}")

Total transcripts inputted from variance matrix: 373
Transcripts successfully mapped to prediction file: 373
Transcripts retained after dropping uniform macro-SCL genes: 277
Total high-confidence candidate genes: 88
Final dataset saved to: Data/FIMO_New/Aginst_MEME_file/Functional_Zipcode_Candidates.csv
